In [ ]:
# 공식 코드 
import torch
from torch.utils import data
from torch import nn
from torch.optim import lr_scheduler
from craft import CRAFT
from S_loss import Loss
import os
import time
import numpy as np
from S_config import cfg
from S_dataset import SynthTextDataset
from S_sync_batchnorm import convert_model

# 저장 경로
SAVE_PATH = "./models/"
# 저장 파일명
SAVE_FILE = "top_model_train_wbs.pth"
# 저장 모델구조 및 파라미터 모두 저장
SAVE_MODEL = "top_model_all.pth"

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)


def train(cfg):
	device = "cpu"
	model = CRAFT()
	model = convert_model(model)
	data_parallel = False
	if torch.cuda.device_count() > 1:
		model = nn.DataParallel(model)
		data_parallel = True
	model.to(device)

	trainset = SynthTextDataset(cfg)
	train_loader = data.DataLoader(trainset, batch_size=cfg.batch_size, shuffle=cfg.shuffle, \
                                   num_workers=cfg.num_workers, drop_last=cfg.drop_last)
	file_num = len(trainset)
	batch_num = int(file_num/cfg.batch_size)
	criterion = Loss()
	optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
	scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=10, verbose=True
    )
	
	loss_history = []
	for epoch in range(cfg.epoch_iter):	
		epoch_loss = 0
		epoch_time = time.time()
		for i, (img, gt_region, gt_affinity, conf_map) in enumerate(train_loader):
			model.train()
			start_time = time.time()
			img, gt_region, gt_affinity, conf_map = list(map(lambda x: x.to(device), [img, gt_region, gt_affinity, conf_map]))
			pred_region, pred_affinity = model(img)
			loss = criterion(gt_region, pred_region, gt_affinity, pred_affinity, conf_map)
			epoch_loss += loss.item()
			optimizer.zero_grad()
			loss.backward()
			optimizer.step()

			print('Epoch is [{}/{}], mini-batch is [{}/{}], time consumption is {:.8f}, batch_loss is {:.8f}'.format(\
              epoch+1, cfg.epoch_iter, i+1, batch_num, time.time()-start_time, loss.item()))
			loss_history.append(loss.item())

			scheduler.step()
			print()
			print(f"scheduler.num_bad_epochs: {scheduler.num_bad_epochs}", end=" ")
			# PyTorch에서 학습률 스케줄러(Scheduler)를 사용할 때, 현재 학습률이 개선되지 않은(epoch의 손실이 향상되지 않은) 연속적인 epoch의 수를 나타내는 변수
			print(f"scheduler.patience: {scheduler.patience}")
			print()
					
			if len(loss_history) == 1:
				# 첫번째라서 무조건 모델 파라미터 저장
				torch.save(model.state_dict(), SAVE_PATH + SAVE_FILE)

				# 모델 전체 저장
				torch.save(model, SAVE_PATH + SAVE_MODEL) 
			
			else:
				if loss_history[-1] <= min(loss_history):
					torch.save(model.state_dict(), SAVE_PATH + SAVE_FILE)
					# 모델 전체 저장
					torch.save(model, SAVE_PATH + SAVE_MODEL)


		print('epoch_loss is {:.8f}, epoch_time is {:.8f}'.format(epoch_loss/batch_num, time.time()-epoch_time))
		print(time.asctime(time.localtime(time.time())))
		print('='*50)


if __name__ == '__main__':
	train(cfg.train)

FileNotFoundError: [Errno 2] No such file or directory: 'MLT/coords_train_230000.txt'